In [ ]:
import anthropic 
from anthropic import BadRequestError
import mysql.connector
from dotenv import load_dotenv
import base64
from dateutil import parser
from pydantic import BaseModel
from pydantic import ValidationError
from typing import Optional
import yaml
from pathlib import Path


import json
import os

load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")
print(anthropic_key)

In [ ]:
# Inserts yaml file into main
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)


In [ ]:
# FUNCTION: Connects to the LLM. Scans the respected pdf. Translates handwritten data into dictionary. 
client = anthropic.Anthropic()
def read_trf(pdf):
    # Convert PDF into base64
    with open(pdf, "rb") as file:
        pdf_bytes = file.read()
    encoded_bytes = base64.b64encode(pdf_bytes)
    pdf_base64_string = encoded_bytes.decode("utf-8")

    # Call Claude, prompt, return data into structured dictionary
    
    client_message = client.messages.create(
        model=config["model"],
        max_tokens=8000,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": pdf_base64_string,
                        },
                    },
                    {
                        "type": "text",
                        "text": config["trf_extraction_prompt"]
                            
                        
                    },
                ],
            }
        ],
    )
    # Claude often wraps its JSON in markdown backticks ('''json''') 
    # Strip them before parsing so json.load() doesnt crash on the markdown backtick
    response_text = ""
    for block in client_message.content:
        if block.type == "text":
            response_text = block.text
            break
    response_text = response_text.strip()
    response_text = response_text.removeprefix("```json").removeprefix("```")
    response_text = response_text.removesuffix("```")
    response_text = response_text.strip()

    try: 
        extracted_data = json.loads(response_text)
    except json.JSONDecodeError:
        return {"status": "failed", "reason": "Could not parse a valid JSON response"}
    
    return extracted_data
    




In [ ]:
# Two pass call read_trf(). Inputs only matching values into the validated JSON.
def two_pass_extraction(pdf):
    try:
        first_pass = read_trf(pdf)
        if first_pass == {"status": "failed", "reason": "Could not parse a valid JSON response"}:
            return {"status": "failed", "reason": "Could not parse a valid JSON response on first pass"}
        second_pass = read_trf(pdf)
        if second_pass == {"status": "failed", "reason": "Could not parse a valid JSON response"}:
            return {"status": "failed", "reason": "Could not parse a valid JSON response on second pass"}
    except BadRequestError as e:
        print({"status": "failed", "reason": str(e)})
        raise
        
    two_pass_valid_fields = {}
    two_pass_invalid_fields = {}

    for field, value in first_pass["extracted_fields"].items():
        second_value = second_pass["extracted_fields"][field]["value"]
        if value["value"] == second_value:
            two_pass_valid_fields[field] = {"value": value["value"]}
        else:
            two_pass_invalid_fields[field] = {
                "first_pass_value": value["value"], 
                "second_pass_value": second_value
            }
    
    return {
                "extracted_fields": two_pass_valid_fields,
                "flagged_fields": two_pass_invalid_fields
            }



    

In [ ]:
# Loops through all files within the folder running the two_pass_extraction() system on every file
def read_all_trf_files(file_path):
    trf_folder = Path(file_path)
    json_extracted_folder_output = {}
    for trf_file in trf_folder.glob("*.pdf"):
        extracted_file = two_pass_extraction(trf_file)
        if extracted_file["status"] == "failed":
            json_extracted_folder_output[trf_file.stem] = {"status": "failed", "reason": f"Could not parse a valid JSON response on {trf_file.stem}"}
        else:
            json_extracted_folder_output[trf_file.stem] = extracted_file
    return json_extracted_folder_output
    

In [ ]:
# Pydantic checker to check JSON schema validation
class Field(BaseModel):
    value: str


class ExtractedFields(BaseModel):
    # Physician Information
    practice_client_name: Optional[Field] = None
    ordering_physician_phone: Optional[Field] = None
    ordering_physician_last_name: Optional[Field] = None
    ordering_physician_first_name: Optional[Field] = None
    npi: Optional[Field] = None
    ordering_physician_email: Optional[Field] = None
    ordering_physician_fax: Optional[Field] = None
    ordering_physician_street_address: Optional[Field] = None
    ordering_physician_building_number: Optional[Field] = None
    ordering_physician_city: Optional[Field] = None
    ordering_physician_state: Optional[Field] = None
    ordering_physician_postal_code: Optional[Field] = None
    # Patient Information
    patient_last_name: Optional[Field] = None
    patient_first_name: Optional[Field] = None
    patient_middle_initial: Optional[Field] = None
    date_of_birth: Optional[Field] = None
    patient_phone: Optional[Field] = None
    sex_at_birth: Optional[Field] = None
    patient_email: Optional[Field] = None
    medical_record_number: Optional[Field] = None
    patient_street_address: Optional[Field] = None
    patient_city: Optional[Field] = None
    patient_state: Optional[Field] = None
    patient_zip_code: Optional[Field] = None
    patient_country: Optional[Field] = None
    # Demographic
    race: Optional[Field] = None
    ethnicity: Optional[Field] = None
    # Patient History
    patient_history_diabetes: Optional[Field] = None
    patient_history_family_heart: Optional[Field] = None
    patient_history_high_dose_biotin: Optional[Field] = None
    # Billing Information
    billing_type: Optional[Field] = None
    # Specimen Collection
    specimen_collection_date: Optional[Field] = None
    specimen_collection_time: Optional[Field] = None
    specimen_collected_by: Optional[Field] = None
    # Physician Signature
    ordering_physician_signature_status: Optional[Field] = None
    ordering_physician_date: Optional[Field] = None
    # Patient Acknowledgment
    patient_acknowledgment_signature_status: Optional[Field] = None
    patient_acknowledgment_date: Optional[Field] = None

class ICD10Page(BaseModel):
    icd10_codes: Optional[list] = None
    icd10_other: Optional[str] = None

In [ ]:
# JSON Outer Shell
class TRFExtraction(BaseModel):
    extracted_fields: ExtractedFields
    flagged_fields: Optional[dict] = None
    icd10: Optional[ICD10Page] = None


In [ ]:
def convert_to_json(extracted_data):

    # Convert all dates to the same MM/DD/YYYY format, if blank returns "blank"
    list_fields_date = ["date_of_birth", "ordering_physician_date", "patient_acknowledgment_date", "specimen_collection_date"]
    for date_types in list_fields_date:
        if date_types in extracted_data["extracted_fields"]:
            try:
                parsed_data = parser.parse(extracted_data["extracted_fields"][date_types]["value"])
                structured_date = parsed_data.strftime("%m/%d/%Y")
                extracted_data["extracted_fields"][date_types]["value"] = structured_date
            except (parser.ParserError, ValueError):
                pass

    # Validation of json fields through Pydantic
    try:
        validated_data = TRFExtraction(**extracted_data)
    except ValidationError as e:
        return {"status": "failed", "reason": str(e)}
    
    normalized_json = validated_data.model_dump()
    
    return normalized_json

In [ ]:
# testing purposes, DELETE AFTER USE
read_trf(config["input_folder"])
result = two_pass_extraction(config["input_folder"])
final = convert_to_json(result)
print(final)